# Unit 4 — Supervised Fine-Tuning (SFT)

> *Part of the [RL for Robotics & LLMs](https://github.com/AliBuildsAI/rl-for-robotics-llms) series.*

Units 1–3 built policy gradients on robots, where the **environment gives a
reward**.  Units 4–8 move to language models, where it does not.  But before any
RL, there is a supervised step that everyone does first: **SFT**.

A *pretrained* language model knows grammar and facts, but it only does one
thing — predict the next token of internet text.  Ask it a question and it may
continue the question, or drift, rather than *answer*.  **Supervised
fine-tuning** teaches it to follow instructions by training on
(instruction, good-response) pairs.

This is the first hands-on stage of the RLHF pipeline, and the model we train
here is the **starting point for the rest of the series** — the reward model in
Unit 5 and the policy we optimise in Units 6–8 all build on an SFT model.

**What we build in this unit:**

| Part | What | Outcome |
|------|------|---------|
| A | Load a *base* LM + an instruction dataset | See the (prompt → response) format |
| B | Train with SFT (TRL), masking the loss to the response only | An instruction-following model |
| C | Before/after generation | Watch the base model start to *answer* instead of ramble |


---
## 1  Where SFT Sits — The Pipeline

A modern chat model is built in three stages:

| Stage | Trains on | Result |
|-------|-----------|--------|
| **1. Pretraining** | Trillions of internet tokens (next-token prediction) | Knows language & facts, but doesn't follow instructions |
| **2. SFT** *(this unit)* | (instruction, response) demonstrations | Follows instructions, as well as the demos |
| **3. RLHF** *(Units 5–8)* | Preferences / rewards | Goes *beyond* the demonstrations |

SFT is the bridge from a raw pretrained model to something that behaves like an
assistant.  It is also the **ceiling-setter for imitation**: the model can only
be as good as the responses it's shown.  That ceiling is exactly why RLHF
exists — to push past "imitate the demo" toward "produce what people actually
prefer."  But you need a competent instruction-follower first, and that's SFT.


---
## 2  What SFT Actually Is — Next-Token Prediction, Curated

Here is the part that surprises people: **SFT uses the exact same loss as
pretraining** — cross-entropy on the next token. Nothing about the objective is
new.  Two things change:

1. **The data.** Instead of random internet text, we train on curated
   (instruction → response) pairs, formatted with the model's **chat template**
   (the special tokens that mark *user* and *assistant* turns).

2. **What we compute the loss on.** We only want the model to learn to produce
   the **response**, not to re-generate the user's prompt.  So we **mask the
   prompt tokens out of the loss** and train only on the assistant's tokens.

That second point is the one genuinely SFT-specific idea, and it's a favourite
interview question. The next section makes it concrete.


### Chat templates — the format SFT trains on

How does a model know where the user stops and the assistant starts?  A **chat
template** wraps each turn in special tokens.  The de-facto standard is **ChatML**
(introduced by OpenAI), with three roles:

- **system** — standing instructions for the assistant; appears once, at the start
- **user** — the human's message
- **assistant** — the model's response

A formatted conversation looks like this (Qwen uses ChatML):

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
How many helicopters can a human eat in one sitting?<|im_end|>
<|im_start|>assistant
```

Two things to note:

- The `<|im_start|>assistant\n` at the end is the **generation prompt** — it's
  where the model begins writing. (`apply_chat_template(..., add_generation_prompt=True)`
  produces exactly this; we use it for inference.)
- Once *all* the SFT (and later preference) data is packed into one consistent
  template, models follow it with near-perfect reliability — which is what makes
  the special tokens (`<|im_end|>`) a clean, learnable "stop" signal.

This template is also exactly what the loss-masking in the next section operates
on: everything up to and including `<|im_start|>assistant\n` is *prompt* (masked);
the assistant's text plus its closing `<|im_end|>` is the *completion* (trained).


---
## 3  The Key Detail — Completion-Only Loss Masking

For a training example, we build one token sequence:

```
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>
```

We feed the whole thing to the model, but for the **labels** we replace every
token belonging to the prompt (the system/user part, up to and including the
`assistant` header) with the ignore-index **`-100`**.  PyTorch's cross-entropy
skips any position whose label is `-100`, so **no gradient flows from the prompt
tokens** — the model is graded only on the response it should have produced.

Why it matters:

- If you *didn't* mask, the model would also be trained to generate user prompts
  — wasting capacity and teaching the wrong behaviour.
- The masked positions still serve as **context** (the model attends to them);
  they just don't contribute to the loss.

In code, TRL's `SFTTrainer` does this for you via `completion_only_loss`. But
we'll first build the masked label tensor **by hand** in §B so you can see the
`-100`s with your own eyes.


---
## 4  Setup

A GPU matters here (transformer forward/backward).  With Nathan's full recipe
(3 epochs over ~9.5K examples at 2048-token length), a **T4 will run it but
slowly (~30–45 min)**; an **A100 finishes in a few minutes**.  Gradient
checkpointing (set below) keeps the 2048-length sequences within T4 memory.  The
next cell auto-detects precision: bf16 on A100/L4, fp16 on T4 (no bf16 support).


In [ ]:
%pip install -q "transformers>=4.45" "trl>=0.12" "datasets>=3.0" accelerate
print("All packages ready.")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__}  |  device: {device}")
if device.type != "cuda":
    print("WARNING: no GPU detected — training will be slow. "
          "Runtime → Change runtime type → GPU.")


### Base model

We deliberately start from **Qwen2.5-0.5B** — the *base* model, **not** the
Instruct version.  That's the whole point: we want to *do* the instruction-tuning
ourselves and watch it work.  (In Unit 5 we'll use an instruction-tuned model,
because there we care about the reward model, not the SFT step.)

`AutoModelForCausalLM` loads the model with its **language-modelling head** intact
(`hidden → vocab`) — unlike Unit 5's reward model, which swapped that head for a
scalar.  Here we keep predicting tokens.


#### The loading arguments

| Argument | What it is | Choice |
|---|---|---|
| `from_pretrained(MODEL_NAME)` | A Hub repo id (or local path); downloads config + weights + tokenizer. | `Qwen/Qwen2.5-0.5B` — the **base** checkpoint |
| `AutoModelForCausalLM` | Loads the model **with** its next-token (LM) head — a generative model. | the right class for SFT (we train it to generate) |
| `torch_dtype=load_dtype` | Precision the weights load in. | bf16 on A100/L4, fp32 on T4 |
| `tokenizer.pad_token = eos_token` | Padding token for batching. | base tokenizers often have none → reuse EOS |


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"   # BASE model — we instruction-tune it ourselves

use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()
use_fp16 = device.type == "cuda" and not use_bf16
load_dtype = torch.bfloat16 if use_bf16 else torch.float32
print(f"bf16: {use_bf16}  |  fp16: {use_fp16}  |  weights: {load_dtype}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=load_dtype)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME}  |  {n_params/1e6:.0f}M params")
print("has chat template:", tokenizer.chat_template is not None)


---
## Part A — The Instruction Data

We use **No Robots** (`HuggingFaceH4/no_robots`) — the same ~9.5K
hand-written instruction dataset Nathan Lambert's reference SFT code uses, and
the one the chapter calls out as an early state-of-the-art set.  Each row has a
`messages` column: a list of `{"role": ..., "content": ...}` turns — exactly the
chat format `SFTTrainer` expects.


#### `load_dataset` arguments

| Argument | What it is | Choice |
|---|---|---|
| `load_dataset("HuggingFaceH4/no_robots")` | Dataset repo id on the Hub. | conversational, already in `messages` format; matches Nathan's SFT recipe |
| `["train"]` | Pick the split. | the full ~9.5K training split (small enough to use whole) |


In [ ]:
dataset = load_dataset("HuggingFaceH4/no_robots")

train_ds = dataset["train"]          # full ~9.5K rows (no subsampling — matches Nathan's recipe)
eval_ds  = dataset["test"]           # held-out ~500 rows — used to measure eval loss
print(f"Train examples: {len(train_ds)}")
print(f"Eval  examples: {len(eval_ds)}")
print(f"Columns: {train_ds.column_names}")


In [ ]:
# Inspect one example: the raw messages, then what the chat template produces.
ex = train_ds[0]
print("=== messages ===")
for m in ex["messages"]:
    print(f"[{m['role']}] {m['content'][:200]}")

print("\n=== after apply_chat_template (what the model actually sees) ===")
print(tokenizer.apply_chat_template(ex["messages"], tokenize=False)[:600])


### How much data? — quality over quantity

A natural question: how many examples do you need?  The history is instructive:

- **~10K hand-written** examples (e.g. the *No Robots* dataset) were
  state-of-the-art early on.
- **LIMA** ("Less Is More for Alignment") showed that as few as **~1,000**
  carefully curated examples can produce a strong instruction-follower — evidence
  that SFT mostly *surfaces* abilities already in the pretrained model rather than
  teaching new ones.
- Today, **large synthetic datasets** (generated by stronger models) work best on
  most tasks.  Around **~1M prompts** is enough for excellent post-training;
  beyond that, **returns diminish quickly**.

Two principles that matter more than raw count:

1. **Quality > quantity.** A small, clean, high-quality set beats a large noisy one.
2. **Distribution match.** The best prompts look like the **downstream tasks you
   care about** — SFT data should resemble what users will actually ask.

There's also a useful robustness fact: *if more stages follow SFT* (reward
modelling, RL), the model can **recover from some noise** in the SFT data — so
SFT doesn't have to be perfect, it has to be a good starting point.  No Robots is
small enough (~9.5K) that we train on the **whole thing**, exactly as Nathan's
reference recipe does.


---
## Part B — Before vs After

First, a helper to generate an answer, and a look at how the **base** model
responds *before* any SFT.  Base models often continue or ramble rather than give
a clean, bounded answer.


In [ ]:
@torch.no_grad()
def generate(prompt, model, max_new_tokens=128):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.pad_token_id)
    # decode only the newly generated tokens (skip the prompt)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:],
                            skip_special_tokens=True)

PROMPTS = [
    "Give me three tips for writing clean code.",
    "Explain what a binary search is in one sentence.",
]

print("=== BASE MODEL (before SFT) ===")
for p in PROMPTS:
    print(f"\nQ: {p}\nA: {generate(p, model)}")


### See the loss mask (the `-100`s)

Before training, let's build the masked labels for one example **by hand** —
the thing §3 described.  We tokenise the full conversation, then the prompt
portion alone, and set the first `len(prompt)` labels to `-100`.


In [ ]:
demo = [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
]

full_text   = tokenizer.apply_chat_template(demo, tokenize=False)
prompt_text = tokenizer.apply_chat_template(demo[:1], tokenize=False,
                                            add_generation_prompt=True)

full_ids   = tokenizer(full_text,   add_special_tokens=False)["input_ids"]
prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

# completion-only labels: -100 over the prompt, real token ids over the response
labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

print(f"{'token':>14} | {'label':>6}")
print("-" * 26)
for tok_id, lab in zip(full_ids, labels):
    tok = tokenizer.decode([tok_id]).replace("\n", "\\n")
    shown = "-100 (masked)" if lab == -100 else str(lab)
    print(f"{tok!r:>14} | {shown:>6}")

masked = sum(l == -100 for l in labels)
print(f"\n{masked}/{len(labels)} tokens masked (the prompt). "
      f"Loss is computed only on the {len(labels)-masked} response tokens.")


### What about multi-turn conversations?

Our demo had one user turn and one assistant turn.  Real conversations have many.
There are two standard masking choices:

1. **Final-turn only** — mask everything except the *last* assistant turn. Simple,
   but you waste the earlier assistant turns as training signal.
2. **Mask user turns only** — mask every user turn, but include **all** assistant
   turns in the loss.  This is the common choice: the model learns from every
   response it should have produced, while never being trained to generate the
   user's messages.

In both cases the rule is the same one idea as before — **never compute loss on a
user/prompt token**.  `SFTTrainer` handles multi-turn masking for you; it's worth
knowing the two options exist because interviewers ask "how do you handle
multi-turn data?"


### Train with TRL's `SFTTrainer`

`SFTTrainer` automates everything above: it applies the chat template, tokenises,
builds the completion-only labels, and runs the standard cross-entropy loss.
We implemented policy gradients ourselves in Units 1–3; from here we use vetted
implementations and focus on understanding what they do.


### Choosing the hyperparameters — what real recipes use

The single most important SFT hyperparameter is the **learning rate**, and the
rule of thumb is sharp: **SFT uses an LR one to two orders of magnitude smaller
than pretraining.** You are *adjusting* a model that already works, not building
one — too high an LR wipes out pretrained knowledge.  Real open recipes:

| Model | Pretraining LR | SFT LR |
|---|---|---|
| OLMo 2 | $3\times10^{-4}$ | $1\times10^{-5}$ |
| OLMo 3 | — | $5\text{–}8\times10^{-5}$ |

Our `1e-5` (sized for the 0.5B) sits right in that band.  A few more
practitioner notes from the book:

- **Schedule:** warm up over a small fraction of steps, then decay (often
  linearly) to near zero.
- **Batch size** is much smaller than pretraining (e.g. OLMo 2 used ~256 prompts
  per step vs 1–2K rows in pretraining).  The *linear scaling rule*: bigger
  batches give lower-variance gradients, which *supports* a higher LR.
- **Epochs:** 1–3 is typical.  More risks memorising the demos.
- **In practice teams sweep a few LRs and pick the best checkpoint on a held-out
  eval** — there's no single magic number.

The meta-point worth saying on camera: *crafting the overall pipeline matters more
than perfecting any one stage.* SFT just needs to be a solid starting point for
the RLHF that follows.


### The eval metric for SFT — held-out loss

What's the right way to know SFT *worked*?  SFT is supervised next-token
prediction, so the natural metric is the **cross-entropy loss on a held-out
split** the model never trained on.  If it drops over the run, the model is
getting better at producing No-Robots-style responses on unseen prompts (lower
loss = lower perplexity = better fit).  We evaluate on the `test` split each
epoch.

(In a full pipeline you'd *also* run an instruction benchmark like IFEval or
MT-Bench for the final quality signal — eval loss is the cheap in-notebook proxy,
and we pair it with the before/after generations below for the qualitative read.)

#### `SFTConfig` arguments

Training knobs follow Nathan Lambert's `sft_olmo2_1b` recipe; the **one
deliberate change** is the learning rate — his `5e-6` was tuned for a 1B model,
so we use **`1e-5`** sized for our 0.5B (still well within the SFT band).  We
also add held-out evaluation + best-checkpoint selection.

| Argument | What it is | Value |
|---|---|---|
| `per_device_train_batch_size=4` · `gradient_accumulation_steps=8` | → **effective batch 32**. | Nathan's |
| `num_train_epochs=3` | Passes over the data. | Nathan's |
| `learning_rate=1e-5` | AdamW step size. | sized for 0.5B (his 1B used `5e-6`) |
| `lr_scheduler_type="linear"` + `warmup_ratio=0.1` | Warm up 10%, then decay. | Nathan's |
| `weight_decay=0.0` · `max_grad_norm=1.0` | Reg + grad clipping. | Nathan's |
| `max_length=2048` | Truncate sequences. | Nathan's |
| `gradient_checkpointing=True` | Recompute activations to save VRAM. | lets 2048 fit |
| `completion_only_loss=True` | **§3 masking** — loss on the response only. | SFT-specific |
| `eval_strategy="epoch"` · `per_device_eval_batch_size=4` | Evaluate on the held-out split each epoch. | the eval metric |
| `save_strategy="epoch"` · `load_best_model_at_end=True` · `metric_for_best_model="eval_loss"` | Keep + restore the lowest-eval-loss checkpoint. | training hygiene |
| `save_total_limit=1` | Keep only the best checkpoint (disk). | Colab-friendly |
| `bf16` / `fp16` · `seed=42` | Mixed precision · reproducibility. | auto / Nathan's |


#### One data-format step: conversational → prompt-completion

No Robots is a **conversational** dataset (a `messages` column).  But
`completion_only_loss=True` tells `SFTTrainer` to expect a **prompt-completion**
dataset — separate `prompt` and `completion` fields — so it knows which tokens to
mask.  So we split each example: everything except the last message is the
`prompt`, and the final assistant turn is the `completion` (the part we train on).
This is the same masking from §3, just expressed in the format the trainer wants.


In [ ]:
def to_prompt_completion(ex):
    msgs = ex["messages"]
    return {"prompt": msgs[:-1], "completion": [msgs[-1]]}

train_pc = train_ds.map(to_prompt_completion, remove_columns=train_ds.column_names)
eval_pc  = eval_ds.map(to_prompt_completion,  remove_columns=eval_ds.column_names)

print("prompt (last turn):", train_pc[0]["prompt"][-1]["content"][:120])
print("completion        :", train_pc[0]["completion"][0]["content"][:120])


In [ ]:
from trl import SFTTrainer, SFTConfig

# Training knobs aligned with Nathan Lambert's rlhf-book SFT config
# (code/instruction_tuning/configs/sft_olmo2_1b.yaml); learning rate sized for 0.5B.
config = SFTConfig(
    output_dir="qwen-sft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,      # effective batch 32
    num_train_epochs=3,
    learning_rate=1e-5,                 # sized for 0.5B (Nathan's 1B used 5e-6)
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    weight_decay=0.0,
    max_grad_norm=1.0,
    max_length=2048,
    gradient_checkpointing=True,        # makes 2048-length fit in less VRAM
    completion_only_loss=True,          # mask the prompt — train on the response only (§3)
    packing=False,
    # ── held-out evaluation + best-checkpoint selection ──
    eval_strategy="epoch",
    per_device_eval_batch_size=4,
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=25,
    report_to="none",
    bf16=use_bf16,
    fp16=use_fp16,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=train_pc,
    eval_dataset=eval_pc,
    processing_class=tokenizer,
)

train_result = trainer.train()
print("\nSFT complete.")

# Show the eval-loss trajectory — the quantitative proof SFT improved.
hist = [(h["epoch"], h["eval_loss"]) for h in trainer.state.log_history if "eval_loss" in h]
print("\nHeld-out eval loss by epoch (should DECREASE):")
for ep, l in hist:
    print(f"  epoch {ep:.0f}:  eval_loss = {l:.4f}")


---
## Part C — After SFT

Same prompts, same `generate` function — now on the fine-tuned model.  Look for:
the answer is **on-topic**, **bounded** (it stops cleanly at `<|im_end|>` instead
of running on), and actually *follows the instruction* rather than continuing it.


In [ ]:
print("=== SFT MODEL (after) ===")
for p in PROMPTS:
    print(f"\nQ: {p}\nA: {generate(p, model)}")


### What to expect — two signals it worked

1. **Quantitative:** the **held-out eval loss printed above should decrease**
   across the 3 epochs.  That's the model fitting unseen No-Robots responses
   better — the metric that says training succeeded.
2. **Qualitative:** the SFT generations should be **cleaner, instruction-shaped,
   and properly terminated** (stop at `<|im_end|>`), where the base model tended
   to ramble, repeat, or not stop.

On a 0.5B model the qualitative change is real but not dramatic — which is why we
pair it with the eval-loss number rather than relying on eyeballing alone.

The honest framing for the video: SFT doesn't make the model *smarter* — the
knowledge was already in the pretrained weights.  It makes the model **use that
knowledge in the format we want** (answer the question, then stop).  Pushing
quality *beyond* the demonstrations is what Units 5–8 are for.


### Key takeaways

The whole unit in seven lines — the things worth saying out loud:

1. **SFT = the same next-token cross-entropy loss as pretraining**, on curated
   (instruction, response) data formatted with a chat template.
2. The one SFT-specific idea is **completion-only loss masking**: prompt tokens get
   label `-100` and contribute *no* gradient — the model is graded on the response.
3. **Chat templates** (ChatML: system / user / assistant) give the model a learnable
   structure and a clean stop token (`<|im_end|>`).
4. **Quality > quantity**: LIMA showed ~1K good examples go a long way; SFT mostly
   *surfaces* pretrained ability, it doesn't add knowledge.
5. **Learning rate is 1–2 orders of magnitude below pretraining** (~`1e-5`–`8e-5`);
   too high erases what the model already knows.
6. **SFT need not be perfect** — later RLHF stages recover from some noise; what
   matters is the *overall* pipeline, not any single stage.
7. SFT sets the **imitation ceiling**; Units 5–8 break past it with preferences and RL.


### Save the checkpoint — this feeds Unit 5

This SFT model is the artifact the rest of the series builds on.  Save it; in
Unit 5 you can point the reward model's `MODEL_NAME` at this folder instead of
the official Instruct checkpoint.


In [ ]:
SAVE_DIR = "qwen-sft-final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved SFT model + tokenizer → {SAVE_DIR}/")
print("In Unit 5: MODEL_NAME = \"" + SAVE_DIR + "\"  (or keep the official Instruct model)")


---
## 5  What's Next — Reward Models (Unit 5)

We now have an instruction-following model — but it only knows how to *imitate*
the responses it was shown.  It has no concept of one answer being **better** than
another.

That's the gap **Unit 5** fills.  We collect human **preference** comparisons
(answer A is better than answer B) and train a **reward model** $r_\phi(x, y)$
that scores any response.  Then in Units 6–8 we use RL — PPO, and later DPO and
GRPO — to push this SFT model *beyond* imitation toward responses people actually
prefer.

→ [Back to the series](https://github.com/AliBuildsAI/rl-for-robotics-llms)
